# ── 1. Cài thư viện ───────────────────────────────────────────────────────────
!pip install -q insightface opencv-python-headless google-api-python-client google-auth
# onnxruntime CPU đã có sẵn trên Colab — không cài onnxruntime-gpu (xung đột CUDA)

In [ ]:
# ── 1. Cài thư viện ───────────────────────────────────────────────────────────
!pip install -q insightface onnxruntime opencv-python-headless google-api-python-client google-auth

In [ ]:
# ── 2. Cấu hình ───────────────────────────────────────────────────────────────
# Mỗi entry là (folder_id, folder_name) — folder_name sẽ hiển thị trong app khi gán sự kiện
FOLDERS = [
    ("PASTE_FOLDER_ID_HERE", "Tên sự kiện / folder"),
    # ("ANOTHER_FOLDER_ID", "Sự kiện khác"),
]
CREDENTIALS_PATH = "/content/credentials.json"
OUTPUT_PATH      = "/content/drive/MyDrive/face_index.json"

In [ ]:
# ── 3. Mount Drive + Upload credentials ───────────────────────────────────────
from google.colab import drive, files
drive.mount('/content/drive')
print("Upload credentials.json:")
files.upload()

In [ ]:
# ── 4. Kết nối Google Drive API ───────────────────────────────────────────────
from google.oauth2 import service_account
from googleapiclient.discovery import build

creds    = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH, scopes=["https://www.googleapis.com/auth/drive.readonly"])
drive_svc = build("drive", "v3", credentials=creds)
print("✅ Drive API connected")

In [ ]:
# ── 5. Load InsightFace ───────────────────────────────────────────────────────
import insightface
face_app = insightface.app.FaceAnalysis(
    name="buffalo_l",
    providers=["CPUExecutionProvider"]
)
face_app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.5)
print("✅ InsightFace loaded")

In [ ]:
# ── 5b. Test InsightFace với 1 ảnh ───────────────────────────────────────────
from google.colab import files as colab_files
import cv2, numpy as np
from IPython.display import display, Image as IPImage

print("Upload 1 ảnh để test:")
uploaded = colab_files.upload()
fname = list(uploaded.keys())[0]

img = cv2.imdecode(np.frombuffer(uploaded[fname], np.uint8), cv2.IMREAD_COLOR)
faces = face_app.get(img)

print(f"\n✅ Phát hiện {len(faces)} khuôn mặt")
for i, face in enumerate(faces):
    x1,y1,x2,y2 = face.bbox.astype(int)
    cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
    cv2.putText(img, f"#{i+1} ({face.det_score:.2f})", (x1, y1-8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    print(f"  Mặt #{i+1}: score={face.det_score:.3f} | embedding 512D ✓")

_, buf = cv2.imencode('.jpg', img)
display(IPImage(data=buf.tobytes()))

In [ ]:
# ── 6. Quét tất cả ảnh (đệ quy thư mục con) ──────────────────────────────────
IMAGE_MIME  = {'image/jpeg', 'image/png', 'image/webp', 'image/heic', 'image/bmp'}
FOLDER_MIME = 'application/vnd.google-apps.folder'

def list_all_recursive(root_id, root_name, _depth=0):
    """Quét đệ quy. Tất cả ảnh đều dùng root_id/root_name của folder gốc."""
    images, subfolders, token = [], [], None
    while True:
        r = drive_svc.files().list(
            q=f"'{root_id}' in parents and trashed=false",
            fields="nextPageToken,files(id,name,mimeType)",
            pageSize=1000, pageToken=token,
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute()
        for f in r.get('files', []):
            if f['mimeType'] == FOLDER_MIME:
                subfolders.append(f)
            elif f['mimeType'] in IMAGE_MIME:
                f['folder_id']   = root_id    # luôn dùng folder gốc
                f['folder_name'] = root_name  # tên để hiển thị trong app
                images.append(f)
        token = r.get('nextPageToken')
        if not token: break

    print('  ' * _depth + f"📁 {root_name}: {len(images)} ảnh, {len(subfolders)} thư mục con")
    for sub in subfolders:
        images += list_all_recursive(root_id, root_name, _depth + 1)  # giữ root_id/name
    return images

all_files = []
for fid, fname in FOLDERS:
    all_files += list_all_recursive(fid, fname)

# Loại trùng
seen = set()
all_files = [f for f in all_files if not (f['id'] in seen or seen.add(f['id']))]
print(f"\n📷 Tổng: {len(all_files)} ảnh")

In [ ]:
# ── 7. Index khuôn mặt → lưu JSON ────────────────────────────────────────────
import io, json, os, cv2, numpy as np
from googleapiclient.http import MediaIoBaseDownload
from tqdm.notebook import tqdm

DRIVE_VIEW = "https://drive.google.com/file/d/{}/view"

def download_image(file_id):
    req = drive_svc.files().get_media(fileId=file_id, supportsAllDrives=True)
    buf = io.BytesIO()
    dl  = MediaIoBaseDownload(buf, req)
    done = False
    while not done: _, done = dl.next_chunk()
    arr = np.frombuffer(buf.getvalue(), np.uint8)
    return cv2.imdecode(arr, cv2.IMREAD_COLOR)

def extract_embedding(img):
    faces = face_app.get(img)
    if not faces: return None
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
    return face.embedding.tolist()

# Resume nếu đã chạy dở
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f: results = json.load(f)
    done_ids = {r['file_id'] for r in results}
    print(f"Resume: bỏ qua {len(done_ids)} ảnh đã xử lý")
else:
    results, done_ids = [], set()

to_process = [f for f in all_files if f['id'] not in done_ids]
print(f"Cần xử lý: {len(to_process)} ảnh")

errors = []
for f in tqdm(to_process):
    try:
        img = download_image(f['id'])
        emb = extract_embedding(img) if img is not None else None
        results.append({
            'file_id':     f['id'],
            'file_name':   f['name'],
            'folder_id':   f['folder_id'],
            'folder_name': f['folder_name'],  # lưu tên để app hiển thị
            'drive_link':  DRIVE_VIEW.format(f['id']),
            'embedding':   emb,
        })
    except Exception as e:
        errors.append({'file': f['name'], 'error': str(e)})

    if len(results) % 50 == 0:
        with open(OUTPUT_PATH, 'w') as out: json.dump(results, out)

with open(OUTPUT_PATH, 'w') as out: json.dump(results, out)

indexed = sum(1 for r in results if r['embedding'])
print(f"\n✅ Xong! {indexed}/{len(results)} ảnh có khuôn mặt | Lỗi: {len(errors)}")
print(f"📁 Đã lưu: {OUTPUT_PATH}")

In [ ]:
# ── 8. Lấy File ID của face_index.json để import vào app ─────────────────────
r = drive_svc.files().list(
    q="name='face_index.json' and trashed=false",
    fields="files(id,name)"
).execute()
for f in r.get('files', []):
    print(f"File ID: {f['id']}")
    print(f"→ Dùng ID này trong app: 'Import từ Colab'")